In [4]:
import random

# одноканальная двумерная свёртка
def convolution_2d(image, kernel):
    h = len(image)
    w = len(image[0])
    kh = len(kernel)
    kw = len(kernel[0])

    pad_h = kh // 2
    pad_w = kw // 2

    result = []
    for i in range(h):
        row = []
        for j in range(w):
            total = 0
            for ki in range(kh):
                for kj in range(kw):
                    ii = i + ki - pad_h
                    jj = j + kj - pad_w
                    if 0 <= ii < h and 0 <= jj < w:
                        total += image[ii][jj] * kernel[ki][kj]
            row.append(total)
        result.append(row)
    return result


# декоратор: применяет одноканальную свёртку к каждому каналу отдельно
def multichannel_filter(func):
    def wrapper(img, kernel):
        h = len(img)
        w = len(img[0])
        c = len(img[0][0])

        first_channel = [[img[i][j][0] for j in range(w)] for i in range(h)]
        first_result = func(first_channel, kernel)

        out_h = len(first_result)
        out_w = len(first_result[0])

        out = [[[0] * c for _ in range(out_w)] for _ in range(out_h)]

        for ch in range(c):
            channel = [[img[i][j][ch] for j in range(w)] for i in range(h)]
            filtered = func(channel, kernel)
            for i in range(out_h):
                for j in range(out_w):
                    out[i][j][ch] = filtered[i][j]

        return out

    return wrapper


# применяем декоратор
@multichannel_filter
def conv2d_channel(img, kernel):
    return convolution_2d(img, kernel)


# пример
M = 5
N = 5
C = 3

img = [[[random.random() for _ in range(C)] for _ in range(N)] for _ in range(M)]
kernel = [[random.random() for _ in range(3)] for _ in range(3)]

print("исходное изображение:")
for ch in range(C):
    print(f"канал {ch}:")
    for i in range(M):
        print(' '.join(f"{img[i][j][ch]:.2f}" for j in range(N)))
    print()

print("ядро свёртки:")
for row in kernel:
    print(' '.join(f"{x:.2f}" for x in row))
print()

res = conv2d_channel(img, kernel)

print("результат поканальной свёртки:")
for ch in range(C):
    print(f"канал {ch}:")
    for i in range(len(res)):
        print(' '.join(f"{res[i][j][ch]:.2f}" for j in range(len(res[0]))))
    print()

исходное изображение (по каналам):
канал 0:
0.93 0.01 0.58 0.40 0.96
0.62 0.20 0.59 0.26 0.50
0.74 0.60 0.93 0.31 0.08
0.48 0.88 0.82 0.41 0.81
0.16 0.99 0.70 0.74 0.57

канал 1:
0.03 0.58 0.67 0.38 0.59
0.43 0.02 0.02 0.73 0.19
0.09 0.46 0.36 0.68 0.12
0.55 0.88 0.08 0.97 0.08
0.99 0.73 0.85 0.26 0.46

канал 2:
0.31 0.81 0.65 0.27 0.57
0.68 0.14 0.87 0.55 0.18
0.29 0.27 0.83 0.69 0.82
0.32 0.78 0.55 0.91 0.91
0.66 1.00 0.25 0.52 0.79

ядро свёртки:
0.20 0.52 0.22
0.41 0.28 0.69
0.44 0.01 0.58

результат поканальной свёртки (по каналам):
канал 0:
0.39 1.41 0.69 1.57 0.55
1.16 1.91 1.28 1.65 0.96
1.51 2.18 1.76 1.72 0.65
1.84 2.17 2.42 2.02 0.83
1.17 1.56 1.81 1.45 0.97

канал 1:
0.43 0.84 1.12 0.92 0.64
0.54 0.91 1.68 1.04 1.04
1.09 0.82 1.89 0.95 0.98
1.35 1.80 1.97 1.46 0.74
1.26 1.79 1.14 1.28 0.47

канал 2:
0.74 1.61 1.10 1.23 0.51
0.79 2.16 1.78 1.88 0.94
1.11 1.64 2.30 2.39 1.13
1.43 1.55 2.49 2.40 1.43
1.21 1.32 1.48 1.58 1.09

